# Features — 피쳐 추출 + 카탈로그 (249종, 전 행 57,477)

**목적**: `origin/` 3분할(train 46,729 / valid 5,000 / holdout 5,748) 전 행에서 산술 전문가의
입력 피쳐 249종을 추출하고, **피쳐 카탈로그**(`feature_catalog.csv`)를 생성한다.
카탈로그가 이후 단계의 진실의 원천이다 — Train 은 카탈로그대로 피쳐를 불러오고,
Ablation 은 카탈로그의 **유형 플래그를 끄는 방식**으로 제거실험을 한다.

| 입력 | 내용 |
|---|---|
| `origin/{train,valid,holdout}.csv` | 원본 텍스트 3분할 |
| **`feature_docs.csv`** | **피쳐 설명의 유일한 원천** — 노트북은 설명을 하드코딩하지 않고 이 파일을 읽는다. 111행 = 형태 5 × 기저 37 + 쌍 8 + BERTScore 12 + 통계 44 + 임베딩 5. 설명을 고치려면 이 파일만 고치면 된다. `gen_docs.py` 가 이 CSV 를 만든다 |

설명은 두 열로 나뉜다 — **`의미`**(무엇을 재는가, 사람 말로)와 **`계산`**(어떻게 계산하는가).
기저를 한 번만 적어두면 카탈로그가 형태와 이어 붙인다:
`의미 = "기저 의미 ▸ 형태 의미"`, `계산 = "기저 계산 ▸ 형태 계산"` (▸ 앞이 응답 1개에 대한 계산, 뒤가 A·B 결합).

| 산출물 | 내용 |
|---|---|
| `features_57k.parquet` | id/role/label/is_tie + 피쳐 249종 (전 행 57,477 수치 행렬). **블록 단위 캐시** — 다시 실행해도 이미 있는 블록은 건너뛴다 |
| `feature_catalog.csv` | **카탈로그** = 문서 + 데이터 판정 병합: feature / 유형 / 의존성 / swap 거동 / 사용(1·0) / 제외사유 / 의미 / 계산. Train·Ablation 이 읽는 진실의 원천 |
| `arith_folds.csv` | train 46,729행 5-fold 물질화 (StratifiedKFold shuffle seed 42) |
| `artifacts/tfidf.joblib` `artifacts/svd.joblib` | **train 행으로만 fit** 한 상태형 아티팩트 |
| `emb_model/` | MiniLM 사본 (없으면 첫 실행 때 HF 자동 다운로드 ~90MB — 인터넷 필요) |

**유형 12종** (각 피쳐는 정확히 하나의 유형에 속한다 — Ablation 의 플래그 단위):

| 유형 | 수 | 내용 |
|---|---|---|
| 길이 | 33 | 문자·단어·문장·토큰 수, 질문 대비 길이, 그 4형·diff 파생 |
| 구조·서식 | 32 | 불릿/번호/헤딩/표/코드/인용/개행/굵게 |
| 가독성·어휘 | 20 | 평균 문장·단어 길이, 어휘 다양성(TTR·3gram·unique) |
| 정보성 | 16 | 숫자/URL/예시/인용 표지 |
| 문체 | 16 | 격식어/헤징/공손/수동태 |
| 감성 | 12 | 긍정/부정/부정문 |
| 태도·기타 | 28 | 거부/호칭/되묻기/비ASCII/빈 응답/이모지 |
| 질문 관련성 | 28 | echo/키워드 커버리지/TF-IDF 코사인/BM25 + **BERTScore 질문 정렬 8**(Q↔A, Q↔B) |
| A/B 유사도 | 12 | 단어·3gram·char3gram 겹침, 길이 균형, 서두/마무리, 숫자 일치 + **BERTScore 4**(A↔B 토큰 정렬) |
| TF-IDF 잠재 | 8 | (A−B) TF-IDF 차이의 SVD 8성분 |
| 임베딩 | 10 | MiniLM cos(Q,A)/cos(Q,B)/cos(A,B) 와 파생 |
| 프롬프트 | 34 | 질문 텍스트 자체의 성질 (q_* + 질문 길이) |

**전개 규칙**: 응답 단위 기저 f 는 `mn_f`(min)/`mx_f`(max)/`ad_f`(|A−B|) — swap 불변 →
**tie축** 재료 — 와 `d_f`(A−B) — swap 부호반전 → **승부축** 재료 — 4형으로 들어간다 (recipe §0).
그래서 `feature_docs.csv` 는 기저를 한 번만 설명하고, 카탈로그가 형태 설명과 합쳐 249행을 만든다.

**구조**: 이 노트북이 train/valid/holdout **전 행을 한 번에 계산**해 `features_57k.parquet`
로 떨어뜨리고, Train·Ablation 은 그 파일을 읽기만 한다(재계산 없음). 수치 행렬만 parquet 인
이유는 249×57,477 float 를 CSV 로 쓰면 ~5배 크고 읽기가 수십 배 느리며 float 왕복에서
값이 흔들리기 때문이다. **사람이 읽고 고치는 것(문서·카탈로그)은 전부 CSV** 다.

**규율**: 상태형(TF-IDF/SVD)은 train 행으로만 fit. 라벨성 컬럼은 메타로만 저장(+AUC 가드).
`train_test_split` 호출 0회. **npz 미사용** — 훈련은 train/valid.csv, 평가는 holdout.csv 만.

**예상 시간**: 첫 실행 **~30–45분** (kw/BM25 루프 ~7분, 임베딩 ~3분, **BERTScore ~12–20분**(512토큰 × Q·A·B 3회 forward), 기저 프레임 ~5분). GPU 기준. **재실행은 블록 캐시로 몇 초** — `REBUILD` 에 넣은 블록만 다시 뽑는다.

In [ ]:
# ── 설정 ──
import os, re, sys, ast, json, math, time
from collections import Counter
import numpy as np
import pandas as pd
from tqdm import tqdm

SEED = 42
N_FEAT = 259            # 통계 44 + 4형·쌍·프롬프트 198 + 임베딩 5 + BERTScore 12
META_COLS = ['id', 'role', 'label', 'is_tie']
EMB_HF_ID = 'sentence-transformers/all-MiniLM-L6-v2'

def _tq(seq, desc):
    return tqdm(seq, desc=desc, file=sys.stdout, smoothing=0, leave=False)

In [ ]:
# ── [정의] 파싱 + 응답 단위 기저 37종 + 쌍 8종 + 4형 전개 ──
def parse_text(x):
    # 원본 셀(문자열화된 리스트 또는 생 문자열) → 평문. json → ast 폴백, None 항 스킵.
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ''
    t = str(x).strip()
    if t.startswith('[') and t.endswith(']'):
        for loader in (json.loads, ast.literal_eval):
            try:
                v = loader(t)
            except Exception:
                continue
            if isinstance(v, list):
                return ' '.join(str(u) for u in v if u is not None)
    return t

def parse_columns(d):
    return ([parse_text(x) for x in d['prompt']],
            [parse_text(x) for x in d['response_a']],
            [parse_text(x) for x in d['response_b']])

_RE_SENT      = re.compile(r'[.!?]+')
_RE_PARA      = re.compile(r'\n\s*\n')
_RE_NUMBERING = re.compile(r'^\s*\d+[.)]\s')
_RE_NUMBER    = re.compile(r'\d+(?:\.\d+)?')
_RE_EXAMPLE   = re.compile(r'for example|for instance|e\.g\.|example:', re.I)
_RE_FOOTNOTE  = re.compile(r'\[\d+\]')
_RE_MDLINK    = re.compile(r'\[[^\]]+\]\([^)]+\)')
_RE_FORMAL    = re.compile(r'\b(?:furthermore|moreover|consequently|therefore|thus|'
                           r'additionally|nevertheless|hence)\b', re.I)
_RE_UNCERTAIN = re.compile(r'\b(?:might|may|perhaps|possibly|likely|probably|seems|appears)\b'
                           r'|i think|not sure', re.I)
_RE_POLITE    = re.compile(r'please|thank|apolog|appreciate|glad to|happy to', re.I)
_RE_PASSIVE   = re.compile(r'\b(?:is|are|was|were|be|been|being)\s+'
                           r'(?:\w+ed|made|done|given|taken|seen|known|found|used|shown|'
                           r'written|built|held|kept|sent|told|thought|understood)\b', re.I)
_RE_REFUSAL   = re.compile(r"as an ai|i can'?t|i cannot|i'?m sorry|i apologize|"
                           r"unable to|sorry,? but", re.I)
_RE_YOU       = re.compile(r'\byou\b', re.I)
_RE_WORD      = re.compile(r'\w+')
_RE_POSITIVE  = re.compile(
    r'\b(?:good|great|excellent|best|better|effective|efficient|helpful|useful|valuable|'
    r'important|easy|simple|powerful|amazing|awesome|wonderful|fantastic|perfect|ideal|'
    r'beneficial|brilliant|impressive|outstanding|remarkable|superior|optimal|convenient|'
    r'flexible|reliable|robust|accurate|correct|clear|safe|secure|fast|quick|smooth|'
    r'love|enjoy|recommended?|advantages?|benefits?|successful?|'
    r'improve(?:d|s|ment)?|enhance[ds]?)\b', re.I)
_RE_NEGATIVE  = re.compile(
    r'\b(?:bad|worse|worst|wrong|errors?|mistakes?|fail(?:s|ed|ure|ures)?|problems?|'
    r'problematic|difficult(?:y|ies)?|hard|harder|impossible|unfortunately|issues?|'
    r'risk(?:s|y)?|danger(?:ous)?|harm(?:ful)?|poor(?:ly)?|lack(?:s|ing)?|incorrect|'
    r'inaccurate|negative|weak|slow|broken|bugs?|confusing|confused|unclear|complicated|'
    r'misleading|flaw(?:s|ed)?|terrible|horrible|awful|useless|invalid|unsafe|insecure|'
    r'unstable|inefficient|disadvantages?|drawbacks?|concern(?:s|ing)?|severe|fatal|'
    r'damage[ds]?|threats?|vulnerable|vulnerabilit(?:y|ies))\b', re.I)
_RE_NEGATION  = re.compile(
    r"\b(?:not|no|never|none|nothing|nobody|nowhere|neither|nor|cannot|without|"
    r"hardly|barely|scarcely|unable)\b|n't", re.I)
_RE_EMOJI     = re.compile('[\U0001F000-\U0001FAFF☀-➿️]')

def _v3_features(t):
    n_char = len(t)
    words = t.split()
    n_w = max(len(words), 1)
    lines = t.split('\n')
    n_sent = len(_RE_SENT.findall(t))
    lower_words = [w.lower() for w in words]
    if len(lower_words) >= 3:
        tri = list(zip(lower_words, lower_words[1:], lower_words[2:]))
        uniq3 = len(set(tri)) / len(tri)
        top3 = Counter(tri).most_common(1)[0][1]      # 최빈 3단어 연쇄의 등장 수
        top_share = top3 / len(tri)
    else:
        uniq3, top3, top_share = 1.0, 0, 0.0
    per100 = 100.0 / n_w
    return {
        'char_len': np.log1p(n_char), 'word_count': np.log1p(len(words)),
        'sentence_count': np.log1p(n_sent),
        'paragraph_count': np.log1p(sum(1 for b in _RE_PARA.split(t) if b.strip())),
        'bullet_count': np.log1p(sum(1 for l in lines if l.lstrip()[:2] in ('- ', '* ', '• '))),
        'numbering_count': np.log1p(sum(1 for l in lines if _RE_NUMBERING.match(l))),
        'heading_count': np.log1p(sum(1 for l in lines if l.lstrip().startswith('#'))),
        'has_table': 1.0 if any('|' in l and '---' in l for l in lines) else 0.0,
        'code_block_count': np.log1p(t.count('```') // 2),
        'quote_count': np.log1p(sum(1 for l in lines if l.lstrip().startswith('>'))),
        'avg_sentence_len': len(words) / max(n_sent, 1),
        'avg_word_len': sum(len(w) for w in words) / n_w,
        'type_token_ratio': len(set(lower_words)) / n_w,
        'unique_3gram_ratio': uniq3,
        'rep_top_share': top_share, 'rep_max_ngram': np.log1p(top3),
        'number_count': np.log1p(len(_RE_NUMBER.findall(t))),
        'url_count': np.log1p(t.count('http')),
        'example_count': np.log1p(len(_RE_EXAMPLE.findall(t))),
        'citation_count': np.log1p(len(_RE_FOOTNOTE.findall(t)) + len(_RE_MDLINK.findall(t))
                                   + t.lower().count('according to') + t.lower().count('source:')),
        'formal_words': len(_RE_FORMAL.findall(t)) * per100,
        'uncertainty_words': len(_RE_UNCERTAIN.findall(t)) * per100,
        'politeness': len(_RE_POLITE.findall(t)) * per100,
        'passive_ratio': len(_RE_PASSIVE.findall(t)) * per100,
        'positive_words': len(_RE_POSITIVE.findall(t)) * per100,
        'negative_words': len(_RE_NEGATIVE.findall(t)) * per100,
        'negation_rate': len(_RE_NEGATION.findall(t)) * per100,
        'refusal': 1.0 if _RE_REFUSAL.search(t) else 0.0,
        'you_rate': len(_RE_YOU.findall(t)) * per100,
        'user_mention': np.log1p(t.lower().count('the user')),
        'newline_count': np.log1p(t.count('\n')),
        'question_marks': np.log1p(t.count('?')),
        'non_ascii_ratio': sum(1 for c in t if ord(c) > 127) / max(n_char, 1),
        'is_empty': 1.0 if not t.strip() else 0.0,
    }

def prompt_echo(p, t):
    pw = set(w.lower() for w in _RE_WORD.findall(p))
    rw = set(w.lower() for w in _RE_WORD.findall(t))
    return len(pw & rw) / len(pw | rw) if pw and rw else 0.0

def _extra_features(t, p):
    pw = set(w.lower() for w in _RE_WORD.findall(p) if len(w) > 2)
    rw = set(w.lower() for w in _RE_WORD.findall(t))
    return {'kw_coverage': (len(pw & rw) / len(pw)) if pw else 0.0,
            'len_vs_prompt': np.log1p(len(t) / (len(p) + 1.0)),
            'bold_count': np.log1p(t.count('**') // 2),
            'emoji_count': np.log1p(len(_RE_EMOJI.findall(t)))}

def extract_unit_frame(texts, prompts=None, desc='기저'):
    # prompts 있으면 39종(V3 34 + prompt_echo + 추가 4), 없으면 34종 (프롬프트 자체용)
    if prompts is None:
        return pd.DataFrame([_v3_features(t) for t in _tq(texts, desc)])
    rows = []
    for t, p in _tq(list(zip(texts, prompts)), desc):
        r = _v3_features(t)
        r['prompt_echo'] = prompt_echo(p, t)
        r.update(_extra_features(t, p))
        rows.append(r)
    return pd.DataFrame(rows)

def _word_set(t):
    return set(w.lower() for w in _RE_WORD.findall(t))

def _word_3grams(t):
    ws = [w.lower() for w in _RE_WORD.findall(t)]
    return set(zip(ws, ws[1:], ws[2:])) if len(ws) >= 3 else set()

def _jaccard(sa, sb):
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

def _tf_cos(ca, cb):
    if not ca or not cb:
        return 0.0
    num = sum(ca[k] * cb[k] for k in set(ca) & set(cb))
    den = (sum(v * v for v in ca.values()) ** .5) * (sum(v * v for v in cb.values()) ** .5)
    return num / den if den else 0.0

def _char3(t, cap=4000):
    t = t[:cap].lower()
    return Counter(t[i:i + 3] for i in range(len(t) - 2)) if len(t) >= 3 else Counter()

def _suffix_ratio(a, b):
    l = min(len(a), len(b))
    if l == 0:
        return 0.0
    k = 0
    while k < l and a[-1 - k] == b[-1 - k]:
        k += 1
    return k / l

def extract_pair_features(a, b):
    la, lb = len(a), len(b)
    lim = min(la, lb)
    k = 0
    while k < lim and a[k] == b[k]:
        k += 1
    sa, sb = set(_RE_NUMBER.findall(a)), set(_RE_NUMBER.findall(b))
    return {'pr_jac_word': _jaccard(_word_set(a), _word_set(b)),
            'pr_jac_3gram': _jaccard(_word_3grams(a), _word_3grams(b)),
            'pr_len_ratio': (min(la, lb) / max(la, lb)) if max(la, lb) else 1.0,
            'pr_prefix_sim': (k / lim) if lim else 0.0,
            'cos_word': _tf_cos(Counter(w.lower() for w in _RE_WORD.findall(a)),
                                Counter(w.lower() for w in _RE_WORD.findall(b))),
            'cos_char3': _tf_cos(_char3(a), _char3(b)),
            'suffix_sim': _suffix_ratio(a, b),
            'num_jac': len(sa & sb) / len(sa | sb) if sa | sb else 0.0}

def build_unit_pair_frame(P, A, B):
    # → 198컬럼 = 4형 전개 156 (기저 39 × mn/mx/ad/d) + 쌍 8 + 프롬프트 q_ 34
    fa = extract_unit_frame(A, P, desc='기저 A')
    fb = extract_unit_frame(B, P, desc='기저 B')
    qf = extract_unit_frame(P, desc='기저 Q').add_prefix('q_')
    out = {}
    for n in fa.columns:
        va, vb = fa[n].to_numpy(np.float32), fb[n].to_numpy(np.float32)
        out[f'mn_{n}'] = np.minimum(va, vb)
        out[f'mx_{n}'] = np.maximum(va, vb)
        out[f'ad_{n}'] = np.abs(va - vb)
        out[f'd_{n}'] = va - vb
    sym = pd.DataFrame(out)
    pair = pd.DataFrame([extract_pair_features(a, b) for a, b in _tq(list(zip(A, B)), '쌍 8종')])
    dfx = pd.concat([sym, pair, qf], axis=1)
    return pd.DataFrame(np.nan_to_num(dfx.to_numpy(np.float32), posinf=0, neginf=0),
                        columns=dfx.columns)
print('정의 완료: 파싱 + 기저 37 + 쌍 8 + 4형 전개')

In [ ]:
# ── [정의] MiniLM 임베딩 — 1회 인코딩으로 파생 10종 ──
def load_embedder(emb_dir='emb_model', device=None):
    import torch
    from transformers import AutoTokenizer, AutoModel
    src = emb_dir if os.path.isdir(emb_dir) else EMB_HF_ID
    tok = AutoTokenizer.from_pretrained(src)
    mod = AutoModel.from_pretrained(src)
    if src != emb_dir:
        tok.save_pretrained(emb_dir)
        mod.save_pretrained(emb_dir)
        print(f'{emb_dir}/ 저장 완료 (오프라인·Kaggle Dataset 동봉용)')
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    return tok, mod.to(device).eval(), device

def encode_texts(tok, mod, texts, device, batch_size=256, max_len=256, max_chars=2000, desc='embed'):
    import torch
    out = []
    for i in _tq(range(0, len(texts), batch_size), desc):
        batch = [(t[:max_chars] if t else ' ') for t in texts[i:i + batch_size]]
        enc = tok(batch, padding=True, truncation=True, max_length=max_len,
                  return_tensors='pt').to(device)
        with torch.no_grad():
            h = mod(**enc).last_hidden_state
        m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
        v = torch.nn.functional.normalize(v, dim=1)
        out.append(v.float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def emb_frames(eq, ea, eb):
    # 같은 배열에서 파생 → (emb_cos, ab_emb_sim) 등 3쌍은 완전중복이 되고 카탈로그에서 한쪽이 사용=0 이 된다
    sqa = (eq * ea).sum(1).astype(np.float32)
    sqb = (eq * eb).sum(1).astype(np.float32)
    sab = (ea * eb).sum(1).astype(np.float32)
    emb5 = pd.DataFrame({'emb_cos': sab, 'emb_q_min': np.minimum(sqa, sqb),
                         'emb_q_max': np.maximum(sqa, sqb), 'emb_q_ad': np.abs(sqa - sqb),
                         'emb_q_d': sqa - sqb}).astype(np.float32)
    emb3 = {'a_emb_sim': sqa, 'b_emb_sim': sqb, 'ab_emb_sim': sab}
    return emb5, emb3

# A·B 토큰 단위 그리디 정렬 (BERTScore) -> bs_prec / bs_rec / bs_f1 / bs_diff.
# 평균 풀링 코사인(emb_cos)은 글을 벡터 하나로 접어 '어느 토큰이 어느 토큰과 대응하는지'를
# 잃는다. 여기서는 토큰 임베딩을 그대로 두고 A x B 코사인 행렬에서 행/열 최댓값을 평균한다
# — 낱말이 달라도 뜻이 같으면 짝이 맺힌다.
#
# 길이 정책이 위의 emb_* 5종과 다르다 (의도적):
#   emb_*      256토큰 · 앞에서 자름 · 2000자 선절단   <- V4 에서 이어받은 값, 계보 유지를 위해 고정
#   BERTScore  512토큰 · 앞 60% + 뒤 40%              <- MiniLM 위치 임베딩 한계까지 씀
# 실측(train 4,000행): 답변의 46%가 256토큰을 넘고 14%가 512를 넘는다. 256이면 긴 답의
# 절반 이하만 보게 되는데, '`A가 말한 것을 B가 담았는가`'를 재는 피쳐에는 치명적이다.
# 문자 단위 선절단은 쓰지 않는다 — 미리 자르면 '뒤 40%'가 진짜 뒤가 아니게 된다.
# (전문 토크나이즈는 전 코퍼스 6초라 비용이 아니다.)
BS_MAX_LEN = 512          # MiniLM 위치 임베딩 한계
BS_HEAD_FRAC = 0.6        # V5 Train_ABE.ipynb 와 같은 비율

def head_tail(ids, k):
    """k 토큰만 남기되 앞 60% + 뒤 40% (V5 Train_ABE.ipynb 와 동일 구현).
    답변은 서두에 결론, 말미에 마무리·요약이 오는 경우가 많아 가운데를 버리는 편이 낫다."""
    ids = list(ids)
    if len(ids) <= k:
        return ids
    h = int(k * BS_HEAD_FRAC)
    t = k - h
    return ids[:h] + ids[-t:] if t > 0 else ids[:h]

def bertscore_features(tok, mod, prompts, texts_a, texts_b, device, batch_size=64,
                       max_len=BS_MAX_LEN, desc='bertscore'):
    """Q·A·B 를 한 배치에서 인코딩해 세 방향의 토큰 정렬을 모두 낸다 (forward 3회, 행렬곱 3회).

      A↔B   bs_prec / bs_rec / bs_f1 / bs_diff       두 답이 서로를 얼마나 덮는가
      Q↔A·B bs_q_cov_*  (질문 토큰 -> 답변)          질문이 요구한 것을 다뤘는가
            bs_q_foc_*  (답변 토큰 -> 질문)          답변이 질문 범위에 머무는가
    A/B 두 값은 4형(min/max/ad/d)으로 접는다 — emb_q_* 와 같은 어법."""
    import torch
    NEG = -1e4                      # -inf 를 쓰면 뒤의 0 곱에서 nan 이 난다
    CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
    BUDGET = max_len - 2            # [CLS], [SEP] 자리

    def _pack(texts):
        # 특수토큰 없이 전문을 토크나이즈 -> head_tail -> 직접 패딩.
        # 빈 응답(이 데이터셋에 실재)은 raw 가 [] 이라 [CLS][SEP] 두 토큰만 남고,
        # 특수토큰을 빼면 유효 토큰 0 -> 아래 ok 가드가 점수를 0 으로 준다.
        raw = tok(list(texts), add_special_tokens=False, verbose=False)['input_ids']
        rows = [[CLS] + head_tail(r, BUDGET) + [SEP] for r in raw]
        L = max(len(r) for r in rows)
        ids = torch.full((len(rows), L), PAD, dtype=torch.long)
        att = torch.zeros((len(rows), L), dtype=torch.long)
        for i, r in enumerate(rows):
            ids[i, :len(r)] = torch.tensor(r, dtype=torch.long)
            att[i, :len(r)] = 1
        return ids.to(device), att.to(device)

    def _tok_emb(texts):
        ids, att = _pack(texts)
        with torch.no_grad():
            h = mod(input_ids=ids, attention_mask=att).last_hidden_state
        h = torch.nn.functional.normalize(h, dim=-1)
        m = att.bool()
        for sid in (CLS, SEP, PAD):     # 특수토큰끼리 무조건 짝지어져 점수가 부푼다
            if sid is not None:
                m &= ids != sid
        return h, m

    def _align(hx, mx, hy, my):
        """(x 토큰이 y 에서 찾은 최대 코사인의 평균, 그 반대). 행렬 한 번으로 양방향."""
        S = torch.bmm(hx, hy.transpose(1, 2))
        nx, ny = mx.sum(1).float(), my.sum(1).float()
        ok = (nx > 0) & (ny > 0)                                  # 빈 응답 방어
        px = S.masked_fill(~my[:, None, :], NEG).max(2).values
        px = (px * mx.float()).sum(1) / nx.clamp(min=1)
        py = S.masked_fill(~mx[:, :, None], NEG).max(1).values
        py = (py * my.float()).sum(1) / ny.clamp(min=1)
        z = torch.zeros_like(px)
        return torch.where(ok, px, z), torch.where(ok, py, z)

    acc = {k: [] for k in ('ab_p', 'ab_r', 'cov_a', 'cov_b', 'foc_a', 'foc_b')}
    for i in _tq(range(0, len(texts_a), batch_size), desc):
        sl = slice(i, i + batch_size)
        hq, mq = _tok_emb(prompts[sl])
        ha, ma = _tok_emb(texts_a[sl])
        hb, mb = _tok_emb(texts_b[sl])
        ab_p, ab_r = _align(ha, ma, hb, mb)       # A->B 정밀도 / B->A 재현율
        foc_a, cov_a = _align(ha, ma, hq, mq)     # A->Q 집중도 / Q->A 커버리지
        foc_b, cov_b = _align(hb, mb, hq, mq)
        for k, v in zip(acc, (ab_p, ab_r, cov_a, cov_b, foc_a, foc_b)):
            acc[k].append(v.float().cpu().numpy())
    v = {k: np.concatenate(x).astype(np.float32) for k, x in acc.items()}

    den = v['ab_p'] + v['ab_r']
    out = {'bs_prec': v['ab_p'], 'bs_rec': v['ab_r'],
           'bs_f1': np.where(den > 1e-6, 2 * v['ab_p'] * v['ab_r'],
                             0.0) / np.where(den > 1e-6, den, 1.0),
           'bs_diff': v['ab_p'] - v['ab_r']}
    for kind in ('cov', 'foc'):                   # 4형 — {min, max, d} 가 {A값, B값} 의 완전 복원 기저
        x, y = v[f'{kind}_a'], v[f'{kind}_b']
        out[f'bs_q_{kind}_min'] = np.minimum(x, y)
        out[f'bs_q_{kind}_max'] = np.maximum(x, y)
        out[f'bs_q_{kind}_ad'] = np.abs(x - y)
        out[f'bs_q_{kind}_d'] = x - y
    return pd.DataFrame({k: w.astype(np.float32) for k, w in out.items()})
print('정의 완료: 임베딩 + BERTScore')

In [ ]:
# ── [정의] 토큰·검색·잠재 통계 44종 — TF-IDF/SVD 만 상태 보유 (train fit) ──
_STAT_DIFF_BASES = ['token_len', 'char_len', 'len_vs_prompt', 'bm25',
                    'keyword_coverage', 'tfidf_sim', 'emb_sim', 'unique_ratio']

def _tok_lens(tok, texts, tag):
    out = np.empty(len(texts), np.float32)
    for i in _tq(range(0, len(texts), 1024), f'token_len {tag}'):
        enc = tok([t if t else ' ' for t in texts[i:i + 1024]], truncation=True, max_length=512)
        out[i:i + 1024] = [len(x) for x in enc['input_ids']]
    return out

def _uniq_ratio(texts):
    out = np.empty(len(texts), np.float32)
    for i, t in enumerate(texts):
        w = _RE_WORD.findall(t.lower())
        out[i] = len(set(w)) / len(w) if w else 0.0
    return out

def _kw_cov(P, R, tag):
    out = np.empty(len(R), np.float32)
    for i in _tq(range(len(R)), f'kw_cov {tag}'):
        q = {w for w in _RE_WORD.findall(P[i].lower()) if len(w) > 2}
        out[i] = len(q & set(_RE_WORD.findall(R[i].lower()))) / len(q) if q else 0.0
    return out

def _bm25(P, A, B):
    bm_a = np.empty(len(P), np.float32)
    bm_b = np.empty(len(P), np.float32)
    for i in _tq(range(len(P)), 'bm25'):
        q_words = [w for w in _RE_WORD.findall(P[i].lower()) if len(w) > 2]
        if not q_words:
            bm_a[i] = bm_b[i] = 0.0
            continue
        doc_a = _RE_WORD.findall(A[i].lower())
        doc_b = _RE_WORD.findall(B[i].lower())
        la, lb = len(doc_a), len(doc_b)
        avg = (la + lb) / 2.0 if (la + lb) > 0 else 1.0
        ca, cb2 = Counter(doc_a), Counter(doc_b)
        sa = sb = 0.0
        for word in set(q_words):          # IDF 코퍼스 = 그 행의 응답 2개 (행 내부, 무상태)
            fa_, fb_ = ca[word], cb2[word]
            dfc = (1 if fa_ > 0 else 0) + (1 if fb_ > 0 else 0)
            idf = math.log((2.0 - dfc + 0.5) / (dfc + 0.5) + 1.0)
            da = fa_ + 1.5 * (1.0 - 0.75 + 0.75 * (la / avg))
            db = fb_ + 1.5 * (1.0 - 0.75 + 0.75 * (lb / avg))
            if da > 0:
                sa += idf * (fa_ * 2.5) / da
            if db > 0:
                sb += idf * (fb_ * 2.5) / db
        bm_a[i] = sa
        bm_b[i] = sb
    return bm_a, bm_b

def build_stat_features(P, A, B, emb3, tok, tfidf=None, svd=None, fit_idx=None):
    # fit_idx 를 주면 TF-IDF/SVD fit(추출), 안 주면 transform(추론) — 코드 경로 하나
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    F_ = {}
    F_['prompt_char_len'] = np.array([len(t) for t in P], np.float32)
    F_['a_char_len'] = np.array([len(t) for t in A], np.float32)
    F_['b_char_len'] = np.array([len(t) for t in B], np.float32)
    F_['prompt_token_len'] = _tok_lens(tok, P, 'P')
    F_['a_token_len'] = _tok_lens(tok, A, 'A')
    F_['b_token_len'] = _tok_lens(tok, B, 'B')
    F_['ratio_token_len'] = (F_['a_token_len'] + 1.0) / (F_['b_token_len'] + 1.0)
    F_['a_len_vs_prompt'] = F_['a_token_len'] / (F_['prompt_token_len'] + 1.0)
    F_['b_len_vs_prompt'] = F_['b_token_len'] / (F_['prompt_token_len'] + 1.0)
    F_['a_unique_ratio'] = _uniq_ratio(A)
    F_['b_unique_ratio'] = _uniq_ratio(B)
    if fit_idx is not None:
        tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2,
                                stop_words='english')
        tfidf.fit([P[i] for i in fit_idx] + [A[i] for i in fit_idx] + [B[i] for i in fit_idx])
    assert tfidf is not None, 'TF-IDF 미적재'
    p_mat, a_mat, b_mat = tfidf.transform(P), tfidf.transform(A), tfidf.transform(B)
    F_['a_tfidf_sim'] = np.asarray(p_mat.multiply(a_mat).sum(axis=1)).ravel().astype(np.float32)
    F_['b_tfidf_sim'] = np.asarray(p_mat.multiply(b_mat).sum(axis=1)).ravel().astype(np.float32)
    F_['a_keyword_coverage'] = _kw_cov(P, A, 'A')
    F_['b_keyword_coverage'] = _kw_cov(P, B, 'B')
    diff_mat = a_mat - b_mat
    if fit_idx is not None:
        svd = TruncatedSVD(n_components=8, random_state=SEED)
        svd.fit(diff_mat[fit_idx])
    assert svd is not None
    sv = svd.transform(diff_mat).astype(np.float32)
    for k in range(8):
        F_[f'tfidf_svd_{k}'] = sv[:, k]
    F_['a_bm25'], F_['b_bm25'] = _bm25(P, A, B)
    F_['a_emb_sim'] = emb3['a_emb_sim']
    F_['b_emb_sim'] = emb3['b_emb_sim']
    F_['ab_emb_sim'] = emb3['ab_emb_sim']
    for name in _STAT_DIFF_BASES:
        dd = (F_[f'a_{name}'] - F_[f'b_{name}']).astype(np.float32)
        F_[f'diff_{name}'] = dd
        F_[f'abs_diff_{name}'] = np.abs(dd)
    out = pd.DataFrame(F_)
    out = pd.DataFrame(np.nan_to_num(out.to_numpy(np.float32), posinf=0, neginf=0),
                       columns=out.columns)
    assert out.shape[1] == 44
    return out, tfidf, svd
print('정의 완료: 토큰·검색·잠재 통계 44종')

In [ ]:
# ── [정의] 문서 로드 — feature_docs.csv 가 이름·설명·블록의 원천이다 ──
# 범주가 곧 추출 블록이다:  형태·기저·쌍 -> unitpair / 통계 -> stat / 임베딩 -> emb / BERTScore -> bertscore
DOC_COLS = ['name', '범주', '유형', '의존성', 'swap', '의미', '계산']
BLOCK_OF = {'형태': 'unitpair', '기저': 'unitpair', '쌍': 'unitpair',
            '통계': 'stat', '임베딩': 'emb', 'BERTScore': 'bertscore'}
docs = pd.read_csv('feature_docs.csv', keep_default_na=False)
assert list(docs.columns) == DOC_COLS, f'feature_docs.csv 스키마 불일치: {list(docs.columns)}'
assert set(docs['범주']) <= set(BLOCK_OF), f"모르는 범주: {set(docs['범주']) - set(BLOCK_OF)}"
BASE = docs[docs['범주'] == '기저'].set_index('name')
FORM = docs[docs['범주'] == '형태'].set_index('name')
DIRECT = docs[~docs['범주'].isin(['기저', '형태'])].set_index('name')     # 이름 그대로 1컬럼
assert len(BASE) + len(FORM) + len(DIRECT) == len(docs)

# 블록별 '이름이 확정된' 컬럼 — 캐시 판정의 기준점.
# unitpair 의 4형 전개 180컬럼은 이름이 코드에서 결정되므로 여기 넣지 않는다. 대신 같은
# 함수가 원자적으로 함께 만들므로 쌍 8종이 있으면 그 블록은 완주한 것이고, 혹시 어긋나도
# 마지막 카탈로그 셀의 양방향 assert 가 잡는다.
BLOCK_COLS = {}
for n, r in DIRECT.iterrows():
    BLOCK_COLS.setdefault(BLOCK_OF[r['범주']], []).append(n)
# mn_/mx_/ad_/d_ x 기저는 이름이 문서만으로 확정된다 — 캐시 판정에 반드시 넣는다.
# (2026-08-03 사고: 기저 2종을 추가했는데 쌍 8종만 보고 낡은 캐시를 통과시켜 249 != 259 로 멈췄다.
#  q_ 는 기저의 부분집합에만 붙어 문서만으로는 못 정하지만, 같은 함수가 원자적으로 만든다.)
BLOCK_COLS['unitpair'] += [f'{p}{b}' for p in ('mn_', 'mx_', 'ad_', 'd_') for b in BASE.index]
print('문서', len(docs), '행 | 블록별 확정 컬럼:',
      {b: len(c) for b, c in BLOCK_COLS.items()}, f'(+ unitpair q_ 전개 {N_FEAT - 225})')


In [ ]:
# ── 데이터 로드 — origin/ 3분할. 훈련=train, ES=valid, 평가=holdout (npz 미사용) ──
def labels_of(d):
    yy = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1,
                    d['winner_tie'] == 1], [0, 1, 2], default=-1).astype(np.int64)
    assert (yy != -1).all(), 'invalid labels!'
    return yy

parts = {}
for name, n_expect in [('train', 46729), ('valid', 5000), ('holdout', 5748)]:
    d = pd.read_csv(os.path.join('origin', f'{name}.csv'))
    assert len(d) == n_expect, f'{name}.csv {len(d)}행 != {n_expect}'
    d['role'] = name
    parts[name] = d
df = pd.concat([parts['train'], parts['valid'], parts['holdout']], ignore_index=True)
assert df['id'].is_unique, 'id 중복!'
y = labels_of(df)
role = df['role'].to_numpy()
m_tr, m_va, m_ho = role == 'train', role == 'valid', role == 'holdout'
y_tr, y_va, y_ho = y[m_tr], y[m_va], y[m_ho]
print(f'train {m_tr.sum():,} / valid {m_va.sum():,} / holdout {m_ho.sum():,}')
print('라벨 분포(a/b/tie):', np.round(np.bincount(y, minlength=3) / len(y), 4))

In [ ]:
# ── 추출 — 블록 단위 캐시 (이미 있는 피쳐는 다시 뽑지 않는다) ──
# 블록 하나가 함수 하나에 대응한다. 캐시(features_57k.parquet)에 그 블록의 컬럼이 다 있으면
# 건너뛰고 그대로 쓴다. 다시 뽑고 싶으면 REBUILD 에 블록 이름을 넣는다 (['all'] 이면 전량).
REBUILD = []                    # 'stat' | 'unitpair' | 'emb' | 'bertscore' | 'all'
CACHE = 'features_57k.parquet'

cached = None
if os.path.exists(CACHE):
    _c = pd.read_parquet(CACHE)
    if len(_c) == len(df) and (_c['id'].to_numpy() == df['id'].to_numpy()).all():
        cached = _c
        print(f'캐시: {CACHE} — {_c.shape[1] - len(META_COLS)}종 (id 정렬 확인)')
    else:
        print(f'캐시 무효 — 행 수/id 가 origin 과 다르다. 전량 재추출한다')

def need(b):
    if 'all' in REBUILD or b in REBUILD:
        return True
    if cached is None:
        return True
    return not set(BLOCK_COLS[b]) <= set(cached.columns)

# 상태형 아티팩트가 없으면 stat 은 캐시가 있어도 다시 뽑아야 한다 (추론 때 필요한 파일이므로)
ART = ['artifacts/tfidf.joblib', 'artifacts/svd.joblib']
if not need('stat') and not all(os.path.exists(p) for p in ART):
    print('  artifacts/ 없음 -> stat 블록 재추출 (TF-IDF·SVD 를 다시 fit 해 저장한다)')
    REBUILD = list(REBUILD) + ['stat']

# stat 은 emb 의 출력(a_emb_sim 등)을 입력으로 먹는다 -> emb 를 다시 뽑으면 stat 도 함께
if need('emb') and not need('stat'):
    print('  emb 재추출 -> stat 도 함께 (a_emb_sim/b_emb_sim/ab_emb_sim 이 emb 출력에서 나온다)')
    REBUILD = list(REBUILD) + ['stat']

plan = {b: ('재추출' if need(b) else '캐시') for b in BLOCK_COLS}
print('블록 계획:', plan)

def take(cols):
    # 캐시 파일에 적힌 순서 그대로 꺼낸다. 문서 순으로 재배열하면 신규 추출과 컬럼 순서가
    # 어긋나고, 완전중복 판정('앞선 컬럼을 남긴다')이 실행마다 달라진다.
    want = set(cols)
    return cached[[c for c in cached.columns if c in want]]

t0 = time.time()
P, A, B = parse_columns(df)
parts = []

# --- MiniLM 이 필요한 블록만 모델을 올린다 ---
if need('emb') or need('bertscore'):
    tok, mod, device = load_embedder('emb_model')
    print('device:', device)
    if need('emb'):
        eq = encode_texts(tok, mod, P, device, desc='embed P')
        ea = encode_texts(tok, mod, A, device, desc='embed A')
        eb = encode_texts(tok, mod, B, device, desc='embed B')
        emb5, emb3 = emb_frames(eq, ea, eb)
        parts.append(emb5)
    else:
        emb5 = take(BLOCK_COLS['emb'])
        emb3 = {c: cached[c].to_numpy(np.float32) for c in ('a_emb_sim', 'b_emb_sim', 'ab_emb_sim')}
        parts.append(emb5)
    if need('bertscore'):
        parts.append(bertscore_features(tok, mod, P, A, B, device))
    else:
        parts.append(take(BLOCK_COLS['bertscore']))
    del mod
    import torch
    if device != 'cpu':
        torch.cuda.empty_cache()
else:
    tok = None
    emb3 = {c: cached[c].to_numpy(np.float32) for c in ('a_emb_sim', 'b_emb_sim', 'ab_emb_sim')}
    parts.append(take(BLOCK_COLS['emb']))
    parts.append(take(BLOCK_COLS['bertscore']))

# --- 통계 44 (토크나이저 + TF-IDF/SVD) ---
if need('stat'):
    from transformers import AutoTokenizer
    tok = tok or AutoTokenizer.from_pretrained('emb_model')
    stat44, tfidf, svd = build_stat_features(P, A, B, emb3, tok, fit_idx=np.where(m_tr)[0])
    import joblib
    os.makedirs('artifacts', exist_ok=True)
    joblib.dump(tfidf, ART[0])
    joblib.dump(svd, ART[1])
    print('artifacts/ 저장 완료 (TF-IDF·SVD 는 train 행으로만 fit)')
    parts.append(stat44)
else:
    parts.append(take(BLOCK_COLS['stat']))

# --- 4형·쌍·프롬프트 188 (무상태 정규식) ---
if need('unitpair'):
    parts.append(build_unit_pair_frame(P, A, B))
else:
    up_cols = [c for c in cached.columns
               if c not in META_COLS and c not in sum(
                   (BLOCK_COLS[b] for b in ('stat', 'emb', 'bertscore')), [])]
    parts.append(cached[up_cols])

F = pd.concat([p.reset_index(drop=True) for p in parts], axis=1)
assert F.shape[1] == N_FEAT, f'{F.shape[1]}종 != {N_FEAT} — 문서와 추출 로직이 어긋났다'
assert not F.columns.duplicated().any(), '컬럼 중복'
assert np.isfinite(F.to_numpy()).all(), 'NaN/inf!'
print(f'{F.shape[0]:,}행 x {F.shape[1]}컬럼  ({(time.time() - t0) / 60:.1f}분)')


In [ ]:
# ── 저장 — parquet(메타 4 + 피쳐 전량) + fold 물질화. 이 파일이 다음 실행의 캐시가 된다 ──
out = F.copy()
out.insert(0, 'id', df['id'].to_numpy())
out.insert(1, 'role', role)
out.insert(2, 'label', y)
out.insert(3, 'is_tie', (y == 2).astype(np.int8))
out.to_parquet('features_57k.parquet')
print(f"saved: features_57k.parquet  {out.shape}  ({os.path.getsize('features_57k.parquet') / 1e6:.1f} MB)")

from sklearn.model_selection import StratifiedKFold
fold = np.full(int(m_tr.sum()), -1, int)
for k, (_, va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED)
                              .split(np.zeros(len(fold)), y_tr)):
    fold[va_i] = k
assert (fold >= 0).all()
pd.DataFrame({'id': df.loc[m_tr, 'id'].to_numpy(), 'fold': fold}).to_csv('arith_folds.csv', index=False)
print(f'saved: arith_folds.csv  fold 크기 {np.bincount(fold).tolist()}')

In [ ]:
# ── 카탈로그 생성 — 설명 원천은 feature_docs.csv (노트북에 설명을 하드코딩하지 않는다) ──
# feature_docs.csv 는 사람이 읽고 고치는 파일이다. 여기서는 읽어서 조립만 한다:
#   형태(mn_/mx_/ad_/d_/q_) x 기저 37 -> 180컬럼, 그 외 69종은 이름 그대로 1:1.
#   의미 = "기저 의미 ▸ 형태 의미",  계산 = "기저 계산 ▸ 형태 계산"  (▸ 앞이 응답 1개 계산, 뒤가 A·B 결합)
# 데이터에 의존하는 판정(분산0 / 완전중복)만 이 셀이 계산해 '사용' 컬럼에 넣는다.
def resolve(c):
    if c in DIRECT.index:                       # 쌍 / 통계 / 임베딩 — 이름 그대로
        r = DIRECT.loc[c]
        return r['유형'], r['의존성'], r['swap'], r['의미'], r['계산']
    for pre, r in FORM.iterrows():              # 형태 접두 + 기저
        base = c[len(pre):]
        if c.startswith(pre) and base in BASE.index:
            b = BASE.loc[base]
            return ((r['유형'] or b['유형']), b['의존성'], r['swap'],
                    f"{b['의미']}  ▸ {r['의미']}", f"{b['계산']}  ▸ {r['계산']}")
    raise AssertionError(f'feature_docs.csv 에 설명이 없다: {c}')

ALL_COLS = list(F.columns)
info = [resolve(c) for c in ALL_COLS]           # 하나라도 없으면 여기서 멈춘다

# 역방향 커버리지 — 문서에만 있고 계산되지 않는 행이 있으면 문서가 낡은 것이다
orphan = sorted(set(DIRECT.index) - set(ALL_COLS))
assert not orphan, f'문서에 있으나 계산되지 않은 피쳐: {orphan}'
used_base = {c[len(p):] for c in ALL_COLS for p in FORM.index
             if c.startswith(p) and c[len(p):] in BASE.index}
assert used_base == set(BASE.index), f'문서의 미사용 기저 행: {sorted(set(BASE.index) - used_base)}'

# 데이터 의존 판정 — 분산0 / 완전중복(앞선 컬럼을 남긴다)
const_cols = [c for c in ALL_COLS if F[c].std() == 0]
seen, dup_of = {}, {}
for c in ALL_COLS:
    if c in const_cols:
        continue
    h = F[c].to_numpy().tobytes()
    if h in seen:
        dup_of[c] = seen[h]
    else:
        seen[h] = c

catalog = pd.DataFrame(info, columns=['유형', '의존성', 'swap', '의미', '계산'])
catalog.insert(0, 'feature', ALL_COLS)
catalog['사용'] = [0 if (c in const_cols or c in dup_of) else 1 for c in ALL_COLS]
catalog['제외사유'] = ['분산0' if c in const_cols
                    else (f'완전중복(={dup_of[c]})' if c in dup_of else '') for c in ALL_COLS]
catalog = catalog[['feature', '유형', '의존성', 'swap', '사용', '제외사유', '의미', '계산']]

EXPECT = {'길이': 33, '구조·서식': 32, '가독성·어휘': 20, '정보성': 16, '문체': 16, '감성': 12,
          '태도·기타': 28, '질문 관련성': 28, 'A/B 유사도': 12, 'TF-IDF 잠재': 8,
          '임베딩': 10, '프롬프트': 36, '반복': 8}   # 반복 = rep_top_share/rep_max_ngram x 4형
got = catalog['유형'].value_counts().to_dict()
assert got == EXPECT, f'유형 개수 불일치: {got}'
assert sum(EXPECT.values()) == N_FEAT == len(catalog)
assert (catalog['의미'].str.len() > 0).all() and (catalog['계산'].str.len() > 0).all()
catalog.to_csv('feature_catalog.csv', index=False)
n_use = int(catalog['사용'].sum())
print(f'saved: feature_catalog.csv — {N_FEAT}종 중 사용 {n_use} / 제외 {N_FEAT - n_use}'
      f' (분산0 {len(const_cols)}, 완전중복 {len(dup_of)})')
if const_cols:
    print('  분산0:', ', '.join(const_cols))
for c, k in dup_of.items():
    print(f'  완전중복: {c} == {k}  (임베딩·키워드를 1회 계산해 공유한 결과)')

In [ ]:
# ── 표 ① 문서 원천 (feature_docs.csv) — 여기를 고치면 카탈로그가 따라 바뀐다 ──
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print('[형태 5] 기저에 곱해지는 접두. 유형이 비면 기저 유형을 상속한다')
    display(FORM.reset_index()[['name', '유형', 'swap', '의미', '계산']])
    print('\n[기저 37] 글 하나에서 계산한다. 형태 x 기저 = 180컬럼')
    display(BASE.reset_index()[['name', '유형', '의미', '계산']])
    print('\n[쌍 8 / BERTScore 12 / 통계 44 / 임베딩 5] 이름 그대로 1컬럼')
    display(DIRECT.reset_index()[['name', '범주', '유형', '의존성', 'swap', '의미', '계산']])

In [ ]:
# ── 표 ② 카탈로그 (전 컬럼) — 문서 + 데이터 판정 병합 결과 ──
print('[유형 x 의존성 요약]')
display(catalog.pivot_table(index='유형', columns='의존성', values='feature',
                            aggfunc='count').fillna(0).astype(int))
print('\n[swap 거동 분포]')
print(catalog['swap'].map(lambda s: s.split(':')[0]).value_counts().to_string())

for g in EXPECT:
    sub = catalog[catalog['유형'] == g].reset_index(drop=True)
    print(f"\n{'=' * 100}\n[{g}] {len(sub)}종 (사용 {int(sub['사용'].sum())})")
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        display(sub[['feature', '의존성', 'swap', '사용', '제외사유', '의미', '계산']])

In [ ]:
# ── 진단 (train 행만) — 라벨 누수 가드 + 단변량 신호 ──
from sklearn.metrics import roc_auc_score
Xtr = F.loc[m_tr].to_numpy(np.float32)
it, nz = (y_tr == 2).astype(int), y_tr != 2
ab = (y_tr[nz] == 0).astype(int)
rows, bad = [], []
for j, c in enumerate(F.columns):
    v = Xtr[:, j]
    if v.std() == 0:
        continue
    a1 = roc_auc_score(it, v)
    a2 = roc_auc_score(ab, v[nz])
    a1, a2 = max(a1, 1 - a1), max(a2, 1 - a2)
    rows.append({'feature': c, 'tie AUC': a1, '승부 AUC': a2})
    if a1 > 0.95 or a2 > 0.95:
        bad.append((c, round(a1, 4), round(a2, 4)))
assert not bad, f'라벨 누수 의심 컬럼: {bad}'   # V4 에서 role/label 잔존으로 AUC 1.0 사고가 났었다
print('leak_guard 통과 — 단일 피쳐 AUC 0.95 초과 없음\n')
diag = pd.DataFrame(rows).merge(catalog[['feature', '유형']], on='feature')
print('[tie축 단변량 상위 15]  (V4 앵커: ad_char_len 0.587 최강)')
display(diag.nlargest(15, 'tie AUC').round(4).reset_index(drop=True))
print('[승부축 단변량 상위 15]')
display(diag.nlargest(15, '승부 AUC').round(4).reset_index(drop=True))

## 다음 단계

1. **[Train.ipynb](Train.ipynb)** — 카탈로그의 사용=1 피쳐로 LGBM·XGBoost·CatBoost 3종 학습
   → `model/` 저장 + `outputs/arith_experts.npz` 로짓 내보내기
2. **[Ablation.ipynb](Ablation.ipynb)** — 카탈로그 유형 플래그를 하나씩 꺼서 제거실험